# 챌린지: 데이터 과학에 관한 텍스트 분석

이 예제에서는 전통적인 데이터 과학 프로세스의 모든 단계를 포함하는 간단한 연습을 해보겠습니다. 코드를 작성할 필요 없이 아래 셀을 클릭하여 실행하고 결과를 관찰하면 됩니다. 도전 과제로서, 다른 데이터로 이 코드를 시도해 보도록 권장합니다.

## 목표

이 수업에서는 데이터 과학과 관련된 다양한 개념들을 논의했습니다. 텍스트 마이닝을 수행하여 관련 개념들을 더 발견해 봅시다. 데이터 과학에 관한 텍스트를 시작으로, 핵심 단어를 추출하고, 그 결과를 시각화해 보겠습니다.

텍스트로는 위키피디아의 데이터 과학 페이지를 사용할 것입니다:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## ជំហានទី 1៖ ទទួលបានទិន្នន័យ

ជំហានដំបូងនៅក្នុងដំណើរការវិទ្យាសាស្ត្រទិន្នន័យគឺការទទួលបានទិន្នន័យ។ យើងនឹងប្រើបណ្ណាល័យ `requests` ដើម្បីធ្វើការនោះ៖


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## ជំហានទី 2៖ ការបម្លែងទិន្នន័យ

ជំហានបន្ទាប់គឺបម្លែងទិន្នន័យឲ្យទៅជារូបមន្តសមរម្យសម្រាប់ការគ្រប់គ្រង។ ក្នុងករណីរបស់យើង យើងបានទាញយកកូដប្រភព HTML ពីទំព័រ ហើយយើងត្រូវបម្លែងវាទៅជាអត្ថបទសាមញ្ញ។

មានវិធីជាច្រើនដែលអាចធ្វើបាន។ យើងនឹងប្រើ [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), ប្រមាញ់បណ្ណាល័យ Python ដែលមានប្រជាប្រិយសម្រាប់ការវិភាគ HTML។ BeautifulSoup អនុញ្ញាតឲ្យយើងបំផ្តោតលើធាតុ HTML ជាក់លាក់ ដូច្នេះ យើងអាចផ្តោតទៅលើមាតិកាចម្បងអត្ថបទពីវិគីភីឌា ហើយកាត់បន្ថយម៉ឺនុយនាវីហ្គេស៊ិន ខាងប័ររបស់ទំព័រ ខាងក្រោម និងមាតិកាដែលគ្មានទាក់ទងផ្សេងទៀត (ទោះបីជាអត្ថបទរចនាសម្ព័ន្ធខ្លះនៅតែមាន)។


ជាដំបូង យើងត្រូវតែដំឡើងបណ្ណាល័យ BeautifulSoup សម្រាប់វិភាគ HTML៖


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## ជំហានទី 3៖ ទទួលបានការយល់ឃើញ

ជំហានសំខាន់បំផុតគឺបម្លែងទិន្នន័យរបស់យើងឱ្យទៅជារូបភាពមួយដែលយើងអាចទាញយកការយល់ឃើញបាន។ ក្នុងករណីរបស់យើង យើងចង់បញ្ចេញពាក្យគន្លឹះពីអត្ថបទ ហើយមើលថាពាក្យគន្លឹះណាដែលមានន័យច្រើនជាងគេ។

យើងនឹងប្រើបណ្ណាល័យ Python ដែលមានឈ្មោះ [RAKE](https://github.com/aneesha/RAKE) សម្រាប់ការបញ្ចេញពាក្យគន្លឹះ។ ជាដំបូង យើងត្រូវតំឡើងបណ្ណាល័យនេះ ប្រសិនបើវាមិនមាននៅក្នុងប្រព័ន្ធ៖ 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

មុខងារសំខាន់សំរាប់ត្រូវបានផ្តល់ពីវត្ថុ `Rake` ដែលយើងអាចផ្លាស់ប្ដូរតាមប៉ារ៉ាម៉ែត្រ។ ក្នុងករណីរបស់យើង យើងនឹងកំណត់រយៈអក្សរតិចជាងទៅរបស់ពាក្យគន្លឹះជា 5 តួអក្សរ ជ្រុលតូចបំផុតនៃការត្រួតពិនិត្យនៃពាក្យគន្លឹះក្នុងឯកសារជា 3 និងចំនួនពាក្យអតិបរមានៅក្នុងពាក្យគន្លឹះជា 2។ អ្នកអាចលេងជាមួយតម្លៃផ្សេងទៀត ហើយសង្កេតមើលលទ្ធផល។


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


យើងបានទទួលបញ្ជីពាក្យជារួចជាមួយនឹងកំរិតសំខាន់ដែលទាក់ទង។ ដូចដែលអ្នកអាចឃើញបាន វិស័យដែលពាក់ព័ន្ធបំផុត ដូចជាការសិក្សាចំណេះដឹងផ្លូវម៉ាស៊ីន និងទិន្នន័យធំ មានតំណែងនៅខាងលើនៃបញ្ជី។

## ជំហានទី ៤៖ ការបង្ហាញលទ្ធផល

មនុស្សអាចបកស្រាយទិន្នន័យបានល្អបំផុតក្នុងរូបរាងដែលអាចមើលឃើញ។ ដូច្នេះ វាធ្វើឱ្យមានអត្ថន័យក្នុងការបង្ហាញទិន្នន័យ ដើម្បីទាញយកមូលដ្ឋានចំឡែកខ្លះ។ យើងអាចប្រើបណ្ណាល័យ `matplotlib` ក្នុង Python ដើម្បីគូរដំណាក់កាលចែកចាយសាមញ្ញនៃពាក្យគន្លឹះជាមួយនឹងពាក់ព័ន្ធរបស់ពួកវា៖


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

ទោះជាយ៉ាងណា មានវិធីមួយល្អជាងនេះសម្រាប់បង្ហាញភាពញឹកញាប់ពាក្យ - ការប្រើប្រាស់ **ពពកពាក្យ**។ យើងគួរតែដំឡើងបណ្ណាល័យមួយផ្សេងទៀតដើម្បីគូរសៀវភៅពពកពាក្យពីបញ្ជីពាក្យគន្លឹះរបស់យើង។ 


In [ ]:
!{sys.executable} -m pip install wordcloud

វត្ថុ `WordCloud` មានការទទួលខុសត្រូវចំពោះការទទួលអត្ថបទដើម ឬបញ្ជីពាក្យដែលបានគណនាមុនជាមួយកម្រិតប្រេកង់របស់ពួកវា ហើយបញ្ជូនត្រឡប់វិប្បភាព ដើម្បីបង្ហាញដោយប្រើ `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

យើងក៏អាចបញ្ជូនអត្ថបទដើមទៅក្នុង `WordCloud` ផងដែរ - មកមើលថាតើយើងអាចទទួលបានលទ្ធផលស្រដៀងគ្នាបានទេ៖


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

អ្នកអាចមើលឃើញថាខ្យល់ពាក្យឥឡូវនេះ​មានរូបរាងគួរឱ្យគាប់ចិត្តច្រើនជាងមុន ប៉ុន្តែវាក៏មានសំឡេងរំខានជាច្រើនផងដែរ (ឧ. ពាក្យមិនទាក់ទងដូចជា `Retrieved on`)។ ក៏ដូចជា យើងបានទទួលពាក្យគន្លឺដែលមានពីរពាក្យច្រើនកម្ដៅតិចជាង ដូចជា *data scientist* ឬ *computer science*។ នេះគឺដោយសារវិធីសាស្រ្ត RAKE ធ្វើការ​ជ្រើសរើសពាក្យគន្លឺបានល្អជាងពីអត្ថបទ។ ឧទាហរណ៍នេះបង្ហាញពីសារៈសំខាន់នៃការព្យួរ និងសំអាត​ទិន្នន័យ ព្រោះរូបភាពច្បាស់នៅចុងបញ្ចប់ នឹងអាចអោយយើងអាចធ្វើសេចក្តីសម្រេចបានល្អជាង។

ក្នុងលំហាត់នេះ យើងបានឆ្លងកាត់ដំណើរការងាយៗមួយនៃការដកស្រង់អត្ថន័យពីអត្ថបទ​វីគីភីឌា ក្នុងទ្រង់ទ្រាយពាក្យគន្លឺ និងខ្យល់ពាក្យ។ ឧទាហរណ៍នេះគឺងាយស្រួល ប៉ុន្តែវាបង្ហាញបានល្អពីជំហានទាំងអស់ដែលអ្នកវិទ្យាសាស្រ្តទិន្នន័យនឹងអនុវត្តនៅពេលធ្វើការ​ជាមួយទិន្នន័យ ដំណើរការចាប់ផ្តើមពីការទទួលទិន្នន័យ រហូតទៅដល់ការបង្ហាញទិន្នន័យ។

ក្នុងវគ្គសិក្សារបស់យើង យើងនឹងពិភាក្សាអំពីជំហានទាំងនោះប្រាប់លម្អិត។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
